# 실습 1-3. FastAPI StreamingResponse 반환

## 실습 목표

이번 실습은 FastAPI의 `StreamingResponse`가 어떻게 동작하는지 코드로 확인하는 단계입니다. generator(또는 async generator)를 넘기면 본문이 청크 단위로 흐른다는 것을 더미 generator로 먼저 확인한 뒤, 같은 자리에 OpenAI Streaming 응답을 끼워 PoC 형태의 스트리밍 엔드포인트까지 만들어 봅니다.

작업 순서는 다음과 같습니다.

- 더미 generator + `StreamingResponse`로 청크가 시간 차이를 두고 흘러나오는 모습 확인
- 받는 쪽 코드(`httpx.Client.stream` + `iter_text`)로 도착하는 대로 출력
- 같은 자리에 OpenAI Streaming 응답을 끼워 PoC 형태의 스트리밍 엔드포인트 구성

> 정식 `/chat/stream` 엔드포인트 설계(Pydantic 요청 모델, 정식 경로 등)는 실습 1-4에서 진행합니다. 이번 실습은 PoC 수준에서 마무리합니다.

## 1. 클라이언트 준비

3일차에서 쓰던 방식 그대로, MLAPI 접속 정보를 `.env`에서 읽습니다. `.env`에 값이 없으면 입력 프롬프트로 받아 `.env` 파일에 저장하고, server.py는 이 `.env`에서 키를 읽습니다. 키가 코드나 server.py에 직접 박히지 않습니다. 포트가 이미 사용 중이면 `PORT`만 바꾸고 셀을 다시 실행하면 됩니다.

In [1]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

base_url   = os.getenv("MLAPI_BASE_URL", "https://mlapi.run/8ebfcef5-8d34-4355-a8d9-89f9253acd2b/v1")
api_key    = os.getenv("MLAPI_API_KEY")
model_name = os.getenv("MLAPI_MODEL", "openai/gpt-5.4")

if not api_key or api_key.startswith("여기에"):
    api_key = getpass("MLAPI API Key를 입력하세요: ")

# server.py가 별도 프로세스에서 읽을 수 있도록 .env 파일에 저장
Path(".env").write_text(
    f"MLAPI_BASE_URL={base_url}\n"
    f"MLAPI_API_KEY={api_key}\n"
    f"MLAPI_MODEL={model_name}\n",
    encoding="utf-8",
)
Path(".gitignore").write_text(".env\n__pycache__/\n", encoding="utf-8")

PORT = 8000                  # 서버가 사용할 포트. 충돌나면 8001 등으로 변경
SERVER_BASE = f"http://localhost:{PORT}"

print("모델명:", model_name, "/ PORT:", PORT)
print("server base:", SERVER_BASE)
print(".env 저장 완료 — server.py가 이 파일에서 키를 읽습니다")

모델명: openai/gpt-5.4 / PORT: 8000
server base: http://localhost:8000
.env 저장 완료 — server.py가 이 파일에서 키를 읽습니다


## 2. server.py 자동 생성

아래 셀을 실행하면 노트북과 같은 폴더에 **server.py** 가 만들어집니다. 이 파일 안에는 두 개의 엔드포인트가 들어 있습니다.

- `/dummy/stream` : 더미 generator 를 StreamingResponse 로 흘리는 엔드포인트. StreamingResponse 자체의 감을 잡기 위한 용도
- `/llm/stream/preview` : OpenAI Streaming 응답을 StreamingResponse 로 그대로 흘리는 PoC. 정식 엔드포인트 설계(`/chat/stream`, 요청 모델 등)는 나중에 진행

이 셀은 **파일을 쓰기만 합니다.** 서버를 띄우지는 않습니다. 서버는 다음 단계에서 터미널로 직접 띄웁니다.

In [2]:
from pathlib import Path

# server.py 베이스 — 기본 엔드포인트 3개. TODO 셀이 이 뒤에 엔드포인트를 덧붙임
server_base = '''import os
import asyncio
from dotenv import load_dotenv
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from openai import OpenAI

load_dotenv()
BASE_URL = os.getenv("MLAPI_BASE_URL")
API_KEY  = os.getenv("MLAPI_API_KEY")
MODEL    = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")

if not BASE_URL:
    raise RuntimeError("MLAPI_BASE_URL이 설정되어 있지 않습니다. .env 파일을 확인하세요.")
if not API_KEY:
    raise RuntimeError("MLAPI_API_KEY가 설정되어 있지 않습니다. .env 파일을 확인하세요.")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

app = FastAPI()


@app.get("/health")
def health():
    return {"ok": True}


# 1) 더미 generator 를 StreamingResponse 로 흘리기
@app.get("/dummy/stream")
async def dummy_stream():
    async def gen():
        for i in range(10):
            await asyncio.sleep(0.2)
            yield f"chunk {i}\\n"
    return StreamingResponse(gen(), media_type="text/plain")


# 2) OpenAI Streaming 응답을 StreamingResponse 로 흘리기 (PoC)
@app.get("/llm/stream/preview")
def llm_stream_preview(message: str = "파이썬 데코레이터를 한 문장으로 설명해줘"):
    def gen():
        stream = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": message}],
            stream=True,
        )
        for chunk in stream:
            if not chunk.choices:
                continue
            piece = getattr(chunk.choices[0].delta, "content", None)
            if not piece:
                continue
            yield piece
    return StreamingResponse(gen(), media_type="text/plain")
'''

Path("server.py").write_text(server_base, encoding="utf-8")
print("server.py 작성 완료 (기본 엔드포인트 3개)")

server.py 작성 완료 (기본 엔드포인트 3개)


## 3. 서버 띄우기

**여기서부터는 노트북이 아니라 터미널에서 직접 실행합니다.** 사용할 명령어를 아래 셀이 출력해주니, 그 한 줄을 그대로 복사해 노트북과 같은 폴더의 터미널에 붙여넣고 실행하세요.

- `--reload` 옵션 덕분에 server.py 를 저장할 때마다 서버가 자동으로 재시작됩니다. 3 단계 셀을 다시 실행하거나, 뒤의 TODO 에서 server.py 를 직접 수정해 저장하면 터미널 로그에 reload 메시지가 뜨고 변경이 반영됩니다.
- 중지하려면 터미널에서 `Ctrl + C`.
- 포트가 이미 쓰이는 중이면 2 단계의 `PORT` 값만 바꿔서 2~4 단계 셀을 다시 실행하세요. 모든 호출 셀이 같은 변수를 참조하므로 자동으로 맞춰집니다.
- 노트북 셀로 서버를 띄우려 하지 말 것 (셀이 블록되어 다음 셀이 안 돌아감).

In [3]:
# 터미널에 붙여넣을 명령어 출력
print(f"uvicorn server:app --reload --port {PORT}")
print()
print("서버 주소:", SERVER_BASE)

uvicorn server:app --reload --port 8000

서버 주소: http://localhost:8000


서버가 떴는지 확인:

In [3]:
import httpx

r = httpx.get(f"{SERVER_BASE}/health", timeout=5.0)
print(r.status_code, r.json())

200 {'ok': True}


## 4. 더미 generator + StreamingResponse 동작 확인

먼저 LLM 없이 **StreamingResponse 자체의 감**을 잡습니다. server.py 의 해당 엔드포인트 본문은 이렇게 생겼습니다.

```python
@app.get("/dummy/stream")
async def dummy_stream():
    async def gen():
        for i in range(10):
            await asyncio.sleep(0.2)
            yield f"chunk {i}\n"
    return StreamingResponse(gen(), media_type="text/plain")
```

핵심은 두 줄입니다.

- `gen()` 은 3-2 에서 다룬 async generator. 0.2 초마다 한 줄씩 `yield` 함.
- `StreamingResponse(gen(), media_type="text/plain")` 가 그것을 HTTP 응답 본문으로 그대로 흘려보냄.

받는 쪽은 `iter_text()` 로 청크 단위로 받습니다. 한꺼번에 받지 않고 도착하는 대로 출력하는 것이 포인트입니다.

In [4]:
import httpx, time

t0 = time.time()
with httpx.Client(timeout=30.0) as c:
    with c.stream("GET", f"{SERVER_BASE}/dummy/stream") as r:
        print("status:", r.status_code)
        for piece in r.iter_text():
            print(f"[{time.time()-t0:.2f}s] {piece!r}")

status: 200
[0.22s] 'chunk 0\n'
[0.42s] 'chunk 1\n'
[0.62s] 'chunk 2\n'
[0.82s] 'chunk 3\n'
[1.02s] 'chunk 4\n'
[1.22s] 'chunk 5\n'
[1.42s] 'chunk 6\n'
[1.62s] 'chunk 7\n'
[1.83s] 'chunk 8\n'
[2.03s] 'chunk 9\n'


출력의 왼쪽 시간을 보면 청크가 0.2 초 간격으로 흘러오는 것이 보입니다. 응답 본문 전체가 한 번에 도착하지 않는다는 것이 `StreamingResponse` 의 정체입니다.

## 5. OpenAI Streaming 응답을 StreamingResponse 로 흘리기 (PoC)

이제 더미 generator 자리에 **OpenAI Streaming 응답**을 끼웁니다. server.py 의 해당 엔드포인트 본문은 이렇게 생겼습니다.

```python
@app.get("/llm/stream/preview")
def llm_stream_preview(message: str = "..."):
    def gen():
        stream = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": message}],
            stream=True,
        )
        for chunk in stream:
            if not chunk.choices:
                continue
            piece = getattr(chunk.choices[0].delta, "content", None)
            if not piece:
                continue
            yield piece
    return StreamingResponse(gen(), media_type="text/plain")
```

감 잡는 포인트:

- 실습 1-1 에서 본 OpenAI `stream=True` 의 chunk 루프를 그대로 가져온 것.
- 한 chunk 안에서 실제 텍스트(`delta.content`) 만 뽑아 `yield`. 이 가공 단계가 실습 1-2 에서 다룬 "변환 async generator" 와 같은 자리.
- StreamingResponse 입장에서는 source 가 더미든 LLM 이든 상관없음. "무엇이든 한 조각씩 흘리는 것" 을 받아 본문으로 흘려보낼 뿐.
- 이 엔드포인트는 어디까지나 PoC 입니다. POST 바디로 요청 받기, Pydantic 요청 모델, 정식 `/chat/stream` 이름, 예외/종료 마커 같은 건 다음 실습에서 다듬습니다.

In [5]:
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "GET",
        f"{SERVER_BASE}/llm/stream/preview",
        params={"message": "파이썬 데코레이터를 한 문장으로 설명해줘"},
    ) as r:
        print("status:", r.status_code)
        for piece in r.iter_text():
            print(piece, end="", flush=True)
print("\n--- end ---")

status: 200
파이썬 데코레이터는 기존 함수를 수정하지 않고도 그 함수의 동작에 기능을 덧붙이거나 감싸는 함수입니다.
--- end ---


응답이 한 번에 큰 덩어리로 떨어지지 않고 토큰 단위로 줄줄 흘러나오면 OK. 더미 generator 자리에 LLM 응답이 들어갔을 뿐, StreamingResponse 의 동작은 단계 5 와 정확히 같습니다.

# TODO

지금까지 본 패턴을 직접 손에 익히기 위한 과제입니다.

## TODO 1. 카운트다운 더미 스트림

server.py 에 새 엔드포인트 `/dummy/countdown` 을 추가합니다.

- GET, 쿼리 파라미터 `start: int = 5`
- async generator 가 `start` 부터 0 까지 0.3 초 간격으로 한 줄씩 `yield`, 마지막에 `"GO!\n"` 한 줄 더
- `media_type="text/plain"` 으로 `StreamingResponse` 반환

아래 셀에서 `todo1` 의 빈칸을 채우고 실행하면 server.py 가 갱신됩니다.

이어서 `--reload` 가 자동 반영 → 호출 셀로 동작을 확인합니다.

In [9]:
# TODO 1 — /dummy/countdown 엔드포인트
todo1 = '''

@app.get("/dummy/countdown")
async def dummy_countdown(start: int = 5):
    async def gen():
        # TODO 1: start 부터 0 까지 0.3초 간격으로 한 줄씩 yield, 끝에 "GO!" 한 줄
        #   (위 /dummy/stream 의 gen 이 형태 참고)
        for i in range(start, -1, -1):
            await asyncio.sleep(0.3)
            yield f"{i}\n"
        yield "GO!"

    return StreamingResponse(gen(), media_type="text/plain")

'''

Path("server.py").write_text(server_base + todo1, encoding="utf-8")
print("server.py 갱신 — /dummy/countdown 추가")

server.py 갱신 — /dummy/countdown 추가


In [10]:
# TODO 1 호출
import httpx, time

t0 = time.time()
with httpx.Client(timeout=30.0) as c:
    with c.stream("GET", f"{SERVER_BASE}/dummy/countdown", params={"start": 5}) as r:
        for piece in r.iter_text():
            print(f"[{time.time()-t0:.2f}s] {piece!r}")

[0.33s] '5\n'
[0.63s] '4\n'
[0.93s] '3\n'
[1.23s] '2\n'
[1.53s] '1\n'
[1.83s] '0\n'
[1.83s] 'GO!'


## TODO 2. LLM 응답 가공해서 흘리기

server.py 에 새 엔드포인트 `/llm/stream/upper` 를 추가합니다.

- GET, 쿼리 파라미터 `message: str`
- OpenAI `stream=True` 호출, 받은 토큰을 `.upper()` 로 대문자화해 흘림
- 빈 chunk 가드(`chunk.choices` / `delta.content`)는 `/llm/stream/preview` 와 동일
- `media_type="text/plain"` 으로 `StreamingResponse` 반환

본문 6단계 `/llm/stream/preview` 와 거의 같고, `yield piece` 한 줄만 `yield piece.upper()` 로 바뀝니다.

아래 셀의 빈칸을 채우고 실행하세요.

In [11]:
# TODO 2 — /llm/stream/upper 엔드포인트
todo2 = '''

@app.get("/llm/stream/upper")
def llm_stream_upper(message: str):
    def gen():
        stream = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": message}],
            stream=True,
        )
        for chunk in stream:
            if not chunk.choices:
                continue
            piece = getattr(chunk.choices[0].delta, "content", None)
            if not piece:
                continue
                
            # TODO 2: piece 를 대문자로 바꿔서 yield
            None

    return StreamingResponse(gen(), media_type="text/plain")
'''

Path("server.py").write_text(server_base + todo1 + todo2, encoding="utf-8")
print("server.py 갱신 — /llm/stream/upper 추가")

server.py 갱신 — /llm/stream/upper 추가


In [12]:
# TODO 2 호출
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "GET",
        f"{SERVER_BASE}/llm/stream/upper",
        params={"message": "explain python decorator in one short sentence"},
    ) as r:
        print("status:", r.status_code)
        for piece in r.iter_text():
            print(piece, end="", flush=True)
print("\n--- end ---")

status: 200
A PYTHON DECORATOR IS A FUNCTION THAT WRAPS ANOTHER FUNCTION OR CLASS TO MODIFY ITS BEHAVIOR WITHOUT CHANGING ITS CODE.
--- end ---


## 정리

- `StreamingResponse(generator, media_type=...)` 한 줄로 FastAPI 응답이 청크 단위로 흐른다. 핵심은 "본문을 한 번에 만들지 않고 generator 를 넘기는 것".
- StreamingResponse 입장에서 source 가 더미 generator 인지 LLM 응답인지는 상관없다. "무엇이든 한 조각씩 흘리는 것" 이면 그대로 본문으로 흘려준다.
- 받는 쪽은 `httpx.Client.stream(...)` + `iter_text()` 로 도착하는 대로 받음. 시간차로 출력이 찍히면 스트리밍이 살아있는 것.
- LLM 결합 시 generator 안에 빈 chunk 가드(`chunk.choices` 비었는지, `delta.content` 비었는지) 가 반드시 필요. 변환·필터 단계도 이 자리에 모아두면 깔끔하다.
- 다음 실습(1-4) 에서 이 PoC 를 정식 `/chat/stream` 엔드포인트(POST + Pydantic 요청 모델 + 예외/종료 마커) 로 다듬는다.